# Event Detection Validation AnalysisThis notebook validates the UPLIFT algorithm's automated event detection against human ground truth annotations for baseball hitting and pitching movements.**Contents:**- Tables 2-6: Summary statistics and reliability metrics- Figures 1-4: Visualizations of validation results

## 1. Setup & Data Loading

In [ ]:
# ============================================================================# IMPORTS AND CONFIGURATION# ============================================================================import numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport seaborn as snsimport pingouin as pgfrom sklearn.metrics import r2_score, mean_absolute_errorimport os# Set plot stylesns.set_style("whitegrid")plt.rcParams['figure.dpi'] = 100# UPLIFT brand colorUPLIFT_PINK = '#EC484F'# Frame rate conversion (240 fps)FRAME_TO_SEC = 1/240print("Libraries imported successfully")

In [ ]:
# ============================================================================
# LOAD DATA FROM CSV FILES
# ============================================================================

# Load main comparison data
hitting_df = pd.read_csv('../data/hitting_data.csv')
pitching_df = pd.read_csv('../data/pitching_data.csv')

# Load inter-rater data
hitting_inter = pd.read_csv('../data/hitting_inter_rater.csv')
pitching_inter = pd.read_csv('../data/pitching_inter_rater.csv')

print(f"✓ Data loaded successfully:")
print(f"  Hitting: {len(hitting_df)} trials (cleaned dataset, auto-failures excluded)")
print(f"  Pitching: {len(pitching_df)} trials (cleaned dataset, auto-failures excluded)")
print(f"  Hitting inter-rater: {len(hitting_inter)} trials (Shun vs Ricky)")
print(f"  Pitching inter-rater: {len(pitching_inter)} trials (Shun vs Ricky)")
print(f"\nNote: Original dataset had 205 hitting trials; 5 were excluded:")
print(f"      • Auto-detection failures (no swings): n=3 [Hit IDs: 7, 9, 184]")
print(f"      • Missing human annotations: n=2 [Hit IDs: 124, 129]")

In [ ]:
# ============================================================================# HELPER FUNCTIONS# ============================================================================def concordance_correlation_coefficient(y_true, y_pred):    """Calculate Lin's Concordance Correlation Coefficient (CCC)."""    y_true = np.array(y_true)    y_pred = np.array(y_pred)        mask = ~(np.isnan(y_true) | np.isnan(y_pred))    y_true, y_pred = y_true[mask], y_pred[mask]    n = len(y_true)        mean_true, mean_pred = np.mean(y_true), np.mean(y_pred)    var_true, var_pred = np.var(y_true, ddof=1), np.var(y_pred, ddof=1)    sd_true, sd_pred = np.std(y_true, ddof=1), np.std(y_pred, ddof=1)        covariance = np.cov(y_true, y_pred, ddof=1)[0, 1]    pearson_r = covariance / (sd_true * sd_pred)        v = sd_pred / sd_true    u = (mean_pred - mean_true) / np.sqrt(sd_pred * sd_true)    Cb = 2 / (v + 1/v + u**2)        ccc = pearson_r * Cb        # 95% CI using Fisher's z-transformation    z = 0.5 * np.log((1 + ccc) / (1 - ccc))    se_z = np.sqrt(1 / (n - 3))    z_lower, z_upper = z - 1.96 * se_z, z + 1.96 * se_z    ci_lower = (np.exp(2 * z_lower) - 1) / (np.exp(2 * z_lower) + 1)    ci_upper = (np.exp(2 * z_upper) - 1) / (np.exp(2 * z_upper) + 1)        return {'CCC': ccc, 'Pearson_r': pearson_r, 'Cb': Cb,             'CI95_lower': ci_lower, 'CI95_upper': ci_upper, 'n': n}def interpret_ccc(ccc):    if ccc >= 0.99: return "Almost Perfect"    elif ccc >= 0.95: return "Substantial"    elif ccc >= 0.90: return "Moderate"    else: return "Poor"print("Helper functions defined")

## 2. Tables

### Table 2: Error Distribution Statistics

In [ ]:
# ============================================================================# TABLE 2: ERROR DISTRIBUTION STATISTICS (Auto vs Human)# ============================================================================# Calculate differences (Human - Auto convention: positive = Auto detected EARLIER)# Hittinghit_fc_diff = hitting_df['Human_Foot_Contact'] - hitting_df['UPLIFT_Foot_Contact']hit_bc_diff = hitting_df['Human_Ball_Contact'] - hitting_df['UPLIFT_Ball_Contact']hit_st_diff = hitting_df['Human_Swing_Through'] - hitting_df['UPLIFT_Swing_Through']# Pitchingpitch_fc_diff = pitching_df['Human_Foot_Contact'] - pitching_df['UPLIFT_Foot_Contact']pitch_rel_diff = pitching_df['Human_Release'] - pitching_df['UPLIFT_Release']# Calculate R² valueshit_fc_r2 = r2_score(hitting_df['Human_Foot_Contact'], hitting_df['UPLIFT_Foot_Contact'])hit_bc_r2 = r2_score(hitting_df['Human_Ball_Contact'], hitting_df['UPLIFT_Ball_Contact'])hit_st_r2 = r2_score(hitting_df['Human_Swing_Through'], hitting_df['UPLIFT_Swing_Through'])pitch_fc_r2 = r2_score(pitching_df['Human_Foot_Contact'], pitching_df['UPLIFT_Foot_Contact'])pitch_rel_r2 = r2_score(pitching_df['Human_Release'], pitching_df['UPLIFT_Release'])# Calculate CCC valueshit_fc_ccc = concordance_correlation_coefficient(hitting_df['Human_Foot_Contact'], hitting_df['UPLIFT_Foot_Contact'])hit_bc_ccc = concordance_correlation_coefficient(hitting_df['Human_Ball_Contact'], hitting_df['UPLIFT_Ball_Contact'])hit_st_ccc = concordance_correlation_coefficient(hitting_df['Human_Swing_Through'], hitting_df['UPLIFT_Swing_Through'])pitch_fc_ccc = concordance_correlation_coefficient(pitching_df['Human_Foot_Contact'], pitching_df['UPLIFT_Foot_Contact'])pitch_rel_ccc = concordance_correlation_coefficient(pitching_df['Human_Release'], pitching_df['UPLIFT_Release'])# Create summary tabletable2_data = []for name, diff, r2, ccc in [    ('Hitting - Foot Contact', hit_fc_diff, hit_fc_r2, hit_fc_ccc),    ('Hitting - Ball Contact', hit_bc_diff, hit_bc_r2, hit_bc_ccc),    ('Hitting - Swing Through', hit_st_diff, hit_st_r2, hit_st_ccc),    ('Pitching - Foot Contact', pitch_fc_diff, pitch_fc_r2, pitch_fc_ccc),    ('Pitching - Release', pitch_rel_diff, pitch_rel_r2, pitch_rel_ccc),]:    table2_data.append({        'Event': name,        'N': len(diff),        'Mean (frames)': f"{diff.mean():.1f}",        'Mean (sec)': f"{diff.mean() * FRAME_TO_SEC:.3f}",        'SD (frames)': f"{diff.std():.1f}",        'Median': f"{diff.median():.0f}",        'MAE (frames)': f"{diff.abs().mean():.1f}",        'Range': f"[{diff.min():.0f}, {diff.max():.0f}]",        'R²': f"{r2:.3f}",        'CCC': f"{ccc['CCC']:.4f}"    })table2_df = pd.DataFrame(table2_data)print("TABLE 2: Error Distribution Statistics (Auto vs Human Ground Truth)")print("="*100)print(table2_df.to_string(index=False))

### Table 3: ICC Inter-Rater Reliability (Overview)

In [ ]:
# ============================================================================# TABLE 3: ICC INTER-RATER RELIABILITY (Shun vs Ricky)# ============================================================================# Prepare data for ICC calculation - long format required# Hittinghitting_icc_data = []for _, row in hitting_inter.iterrows():    hitting_icc_data.append({'ID': row['ID'], 'Rater': 'Shun',                              'Foot_Contact': row['Shun_Foot_Contact'],                             'Ball_Contact': row['Shun_Ball_Contact'],                             'Swing_Through': row['Shun_Swing_Through']})    hitting_icc_data.append({'ID': row['ID'], 'Rater': 'Ricky',                              'Foot_Contact': row['Ricky_Foot_Contact'],                             'Ball_Contact': row['Ricky_Ball_Contact'],                             'Swing_Through': row['Ricky_Swing_Through']})hitting_icc_df = pd.DataFrame(hitting_icc_data)# Pitchingpitching_icc_data = []for _, row in pitching_inter.iterrows():    pitching_icc_data.append({'ID': row['ID'], 'Rater': 'Shun',                              'Foot_Contact': row['Shun_Foot_Contact'],                              'Release': row['Shun_Release']})    pitching_icc_data.append({'ID': row['ID'], 'Rater': 'Ricky',                              'Foot_Contact': row['Ricky_Foot_Contact'],                              'Release': row['Ricky_Release']})pitching_icc_df = pd.DataFrame(pitching_icc_data)# Calculate ICC for each eventprint("TABLE 3: ICC Scores - Inter-Rater Reliability (Shun vs Ricky)")print("="*80)print("ICC Model: Two-way mixed-effects, absolute agreement (ICC3)")print()icc_results = []# Hitting eventsfor event in ['Foot_Contact', 'Ball_Contact', 'Swing_Through']:    icc = pg.intraclass_corr(data=hitting_icc_df, targets='ID', raters='Rater', ratings=event)    icc3 = icc[icc['Type'] == 'ICC3']['ICC'].values[0]    ci = icc[icc['Type'] == 'ICC3'][['CI95%']].values[0][0]    icc_results.append({'Movement': 'Hitting', 'Event': event.replace('_', ' '),                         'ICC': f"{icc3:.4f}", '95% CI': f"[{ci[0]:.4f}, {ci[1]:.4f}]",                        'N Trials': len(hitting_inter)})# Pitching eventsfor event in ['Foot_Contact', 'Release']:    icc = pg.intraclass_corr(data=pitching_icc_df, targets='ID', raters='Rater', ratings=event)    icc3 = icc[icc['Type'] == 'ICC3']['ICC'].values[0]    ci = icc[icc['Type'] == 'ICC3'][['CI95%']].values[0][0]    icc_results.append({'Movement': 'Pitching', 'Event': event.replace('_', ' '),                        'ICC': f"{icc3:.4f}", '95% CI': f"[{ci[0]:.4f}, {ci[1]:.4f}]",                        'N Trials': len(pitching_inter)})icc_table = pd.DataFrame(icc_results)print(icc_table.to_string(index=False))print()print("ICC Interpretation: >0.90 = Excellent, 0.75-0.90 = Good, 0.50-0.75 = Moderate, <0.50 = Poor")

### Table 4: ICC Scores (4 Decimal Places)

In [ ]:
# ============================================================================# TABLE 4: ICC VALUES WITH FULL PRECISION# ============================================================================# Store full precision valuesicc_full_precision = []# Hitting eventsfor event in ['Foot_Contact', 'Ball_Contact', 'Swing_Through']:    icc = pg.intraclass_corr(data=hitting_icc_df, targets='ID', raters='Rater', ratings=event)    icc3_row = icc[icc['Type'] == 'ICC3']    icc_val = icc3_row['ICC'].values[0]    ci = icc3_row['CI95%'].values[0]    pval = icc3_row['pval'].values[0]    icc_full_precision.append({        'Category': 'Hitting',         'Event': event.replace('_', ' '),        'ICC Score': f"{icc_val:.4f}",        '95% CI': f"[{ci[0]:.4f}, {ci[1]:.4f}]",        'p-value': '< 0.001' if pval < 0.001 else f"{pval:.4f}"    })# Pitching eventsfor event in ['Foot_Contact', 'Release']:    icc = pg.intraclass_corr(data=pitching_icc_df, targets='ID', raters='Rater', ratings=event)    icc3_row = icc[icc['Type'] == 'ICC3']    icc_val = icc3_row['ICC'].values[0]    ci = icc3_row['CI95%'].values[0]    pval = icc3_row['pval'].values[0]    icc_full_precision.append({        'Category': 'Pitching',        'Event': event.replace('_', ' '),        'ICC Score': f"{icc_val:.4f}",        '95% CI': f"[{ci[0]:.4f}, {ci[1]:.4f}]",        'p-value': '< 0.001' if pval < 0.001 else f"{pval:.4f}"    })table4_df = pd.DataFrame(icc_full_precision)print("TABLE 4: ICC Values (Inter-Rater Reliability)")print("="*80)print(table4_df.to_string(index=False))

### Table 5: CCC Auto vs Human

In [ ]:
# ============================================================================# TABLE 5: CCC VALUES (Auto vs Human Ground Truth)# ============================================================================table5_data = []for name, ccc_result in [    ('Hitting - Foot Contact', hit_fc_ccc),    ('Hitting - Ball Contact', hit_bc_ccc),    ('Hitting - Swing Through', hit_st_ccc),    ('Pitching - Foot Contact', pitch_fc_ccc),    ('Pitching - Release', pitch_rel_ccc),]:    movement, event = name.split(' - ')    table5_data.append({        'Movement': movement,        'Event': event,        'N': ccc_result['n'],        'CCC': f"{ccc_result['CCC']:.4f}",        'Pearson r': f"{ccc_result['Pearson_r']:.4f}",        'Bias (Cb)': f"{ccc_result['Cb']:.4f}",        '95% CI': f"[{ccc_result['CI95_lower']:.4f}, {ccc_result['CI95_upper']:.4f}]",        'Interpretation': interpret_ccc(ccc_result['CCC'])    })table5_df = pd.DataFrame(table5_data)print("TABLE 5: Concordance Correlation Coefficient (CCC) - Auto vs Human")print("="*100)print("CCC = Pearson_r × Cb (bias correction factor)")print()print(table5_df.to_string(index=False))

### Table 6: CCC Inter-Rater (Shun vs Ricky)

In [ ]:
# ============================================================================# TABLE 6: CCC VALUES (Inter-Rater: Shun vs Ricky)# ============================================================================table6_data = []# Hitting inter-rater CCCfor event in ['Foot_Contact', 'Ball_Contact', 'Swing_Through']:    shun_col = f'Shun_{event}'    ricky_col = f'Ricky_{event}'    ccc_result = concordance_correlation_coefficient(hitting_inter[shun_col], hitting_inter[ricky_col])    table6_data.append({        'Movement': 'Hitting',        'Event': event.replace('_', ' '),        'N': ccc_result['n'],        'CCC': f"{ccc_result['CCC']:.4f}",        'Pearson r': f"{ccc_result['Pearson_r']:.4f}",        'Bias (Cb)': f"{ccc_result['Cb']:.4f}",        '95% CI': f"[{ccc_result['CI95_lower']:.4f}, {ccc_result['CI95_upper']:.4f}]"    })# Pitching inter-rater CCCfor event in ['Foot_Contact', 'Release']:    shun_col = f'Shun_{event}'    ricky_col = f'Ricky_{event}'    ccc_result = concordance_correlation_coefficient(pitching_inter[shun_col], pitching_inter[ricky_col])    table6_data.append({        'Movement': 'Pitching',        'Event': event.replace('_', ' '),        'N': ccc_result['n'],        'CCC': f"{ccc_result['CCC']:.4f}",        'Pearson r': f"{ccc_result['Pearson_r']:.4f}",        'Bias (Cb)': f"{ccc_result['Cb']:.4f}",        '95% CI': f"[{ccc_result['CI95_lower']:.4f}, {ccc_result['CI95_upper']:.4f}]"    })table6_df = pd.DataFrame(table6_data)print("TABLE 6: CCC Values (Inter-Rater Reliability: Shun vs Ricky)")print("="*100)print(table6_df.to_string(index=False))

## 3. Figures

### Figure 1: Error Distribution Histograms

In [ ]:
# ============================================================================
# FIGURE 1: COMBINED VISUALIZATION - ALL EVENTS (3x2 Grid)
# ============================================================================
# Using Auto - Human convention: Positive = Auto detected LATER than human

# Calculate hitting differences (Human - Auto)
# Positive means Auto detected EARLIER than human
hit_fc_diff = hitting_df['Human_Foot_Contact'] - hitting_df['UPLIFT_Foot_Contact']
hit_bc_diff = hitting_df['Human_Ball_Contact'] - hitting_df['UPLIFT_Ball_Contact']
hit_st_diff = hitting_df['Human_Swing_Through'] - hitting_df['UPLIFT_Swing_Through']

# Calculate pitching differences
pitch_fc_diff = pitching_df['Human_Foot_Contact'] - pitching_df['UPLIFT_Foot_Contact']
pitch_rel_diff = pitching_df['Human_Release'] - pitching_df['UPLIFT_Release']

# Create figure
fig, axes = plt.subplots(3, 2, figsize=(16, 12))

# COLUMN 1: HITTING EVENTS
# Negate the differences to flip from (Human - Auto) to (Auto - Human)
hitting_diffs = [(-hit_fc_diff, 'Hitting: Foot Contact'),
                 (-hit_bc_diff, 'Hitting: Ball Contact'),
                 (-hit_st_diff, 'Hitting: Swing Through')]

# Calculate bins with width of 5 frames for range -80 to 80
bins_5_frame = np.arange(-80, 85, 5)

for i, (diff, title) in enumerate(hitting_diffs):
    axes[i, 0].hist(diff, bins=bins_5_frame, alpha=0.7, color='dimgray', edgecolor='black')
    axes[i, 0].axvline(x=0, color='black', linestyle='--', linewidth=4, label='Perfect Agreement')
    axes[i, 0].axvline(x=diff.mean(), color=UPLIFT_PINK, linestyle='-', linewidth=4, label=f'Mean: {diff.mean():.1f}')
    axes[i, 0].set_title(title, fontweight='bold', fontsize=21)
    axes[i, 0].set_xlabel('Relative Timing (frames)', fontsize=18)
    axes[i, 0].set_ylabel('Count', fontsize=18)
    axes[i, 0].set_xlim(-80, 80)
    axes[i, 0].tick_params(axis='both', labelsize=15)
    axes[i, 0].legend(fontsize=15)
    axes[i, 0].grid(False)

# COLUMN 2: PITCHING EVENTS
# Negate the differences to flip from (Human - Auto) to (Auto - Human)
pitching_diffs = [(-pitch_fc_diff, 'Pitching: Foot Contact'),
                  (-pitch_rel_diff, 'Pitching: Release')]

for i, (diff, title) in enumerate(pitching_diffs):
    axes[i, 1].hist(diff, bins=bins_5_frame, alpha=0.7, color='dimgray', edgecolor='black')
    axes[i, 1].axvline(x=0, color='black', linestyle='--', linewidth=4, label='Perfect Agreement')
    axes[i, 1].axvline(x=diff.mean(), color=UPLIFT_PINK, linestyle='-', linewidth=4, label=f'Mean: {diff.mean():.1f}')
    axes[i, 1].set_title(title, fontweight='bold', fontsize=21)
    axes[i, 1].set_xlabel('Relative Timing (frames)', fontsize=18)
    axes[i, 1].set_ylabel('Count', fontsize=18)
    axes[i, 1].set_xlim(-80, 80)
    axes[i, 1].tick_params(axis='both', labelsize=15)
    axes[i, 1].legend(fontsize=15)
    axes[i, 1].grid(False)

# Hide bottom right subplot
axes[2, 1].axis('off')

plt.tight_layout()
plt.savefig('../figures/figure1_error_distribution.png', dpi=300, bbox_inches='tight')
print("✓ Saved: figure1_error_distribution.png")
plt.show()

### Figure 2: Tolerance Interval Proportions

Distribution of detection accuracy across tolerance thresholds for **n=200 hitting trials** and **n=108 pitching trials**.

In [ ]:
# ============================================================================
# FIGURE 2: TOLERANCE INTERVALS - PROPORTION WITHIN THRESHOLD ANALYSIS
# ============================================================================

# Define colors per user specification (4 categories)
COLOR_VERY_ACCURATE = '#00C896'  # Bright teal/green for Very Accurate (0-6 frames)
COLOR_ACCURATE = '#02A173'       # Green for Accurate (7-12 frames)
COLOR_MODERATE = '#596DCA'       # Blue for Moderate (13-24 frames)
COLOR_INACCURATE = '#000000'     # Black for Inaccurate (>24 frames)

# Event data mapping (3x2 layout - Column 1: Hitting, Column 2: Pitching)
fig, axes = plt.subplots(3, 2, figsize=(16, 12))

event_data = [
    (hit_fc_diff, 'Hitting: Foot Contact', axes[0, 0]),
    (pitch_fc_diff, 'Pitching: Foot Contact', axes[0, 1]),
    (hit_bc_diff, 'Hitting: Ball Contact', axes[1, 0]),
    (pitch_rel_diff, 'Pitching: Release', axes[1, 1]),
    (hit_st_diff, 'Hitting: Swing Through', axes[2, 0]),
    (None, '', axes[2, 1])  # Empty subplot
]

for diff_data, event_name, ax in event_data:
    if diff_data is None:
        ax.axis('off')
        continue
        
    abs_diff = np.abs(diff_data)
    total = len(abs_diff)
    
    # Calculate proportions with 4 categories
    very_accurate = (abs_diff <= 6).sum()                              # 0-6 frames
    accurate = ((abs_diff > 6) & (abs_diff <= 12)).sum()               # 7-12 frames
    moderate = ((abs_diff > 12) & (abs_diff <= 24)).sum()              # 13-24 frames
    inaccurate = (abs_diff > 24).sum()                                 # >24 frames
    
    pct_very_accurate = (very_accurate / total) * 100
    pct_accurate = (accurate / total) * 100
    pct_moderate = (moderate / total) * 100
    pct_inaccurate = (inaccurate / total) * 100
    
    # Convert frame thresholds to seconds for labels
    sec_6f = 6 * FRAME_TO_SEC     # 0.025 s
    sec_12f = 12 * FRAME_TO_SEC   # 0.050 s
    sec_24f = 24 * FRAME_TO_SEC   # 0.100 s
    
    # Create bar chart with 4 categories
    categories = [f'Very Accurate\n(0-6 frames)\n(0-{sec_6f:.3f} s)', 
                  f'Accurate\n(7-12 frames)\n({sec_6f:.3f}-{sec_12f:.3f} s)', 
                  f'Moderate\n(13-24 frames)\n({sec_12f:.3f}-{sec_24f:.3f} s)',
                  f'Inaccurate\n(>24 frames)\n(>{sec_24f:.3f} s)']
    values = [pct_very_accurate, pct_accurate, pct_moderate, pct_inaccurate]
    colors = [COLOR_VERY_ACCURATE, COLOR_ACCURATE, COLOR_MODERATE, COLOR_INACCURATE]
    
    bars = ax.bar(categories, values, color=colors, edgecolor='black', alpha=0.7)
    
    # Add percentage labels on top of bars
    for bar, val in zip(bars, values):
        if val > 0:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, 
                    f'{val:.1f}%', ha='center', va='bottom', 
                    fontweight='bold', fontsize=12)
    
    # Styling to match Figure 1
    ax.set_title(event_name, fontweight='bold', fontsize=21)
    ax.set_ylabel('Percentage of Trials (%)', fontsize=18)
    ax.set_ylim([0, 105])
    ax.tick_params(axis='both', labelsize=11)
    ax.grid(False)
    
    # Add n value
    ax.text(0.98, 0.98, f'n = {total}', transform=ax.transAxes, 
            fontsize=15, verticalalignment='top', horizontalalignment='right')

plt.tight_layout()
plt.savefig('../figures/figure2_tolerance_intervals.png', dpi=300, bbox_inches='tight')
print("✓ Saved: figure2_tolerance_intervals.png")
plt.show()

### Figure 3: CCC Concordance Plots

In [ ]:
# ============================================================================
# FIGURE 3: CCC CONCORDANCE PLOTS (Auto vs Human)
# ============================================================================
# Scatter plots showing agreement between automated and human annotations
# with the line of perfect concordance (45° line)

# Create figure with 3x2 grid
fig, axes = plt.subplots(3, 2, figsize=(14, 15))

# Color scheme
SCATTER_COLOR = '#4C72B0'  # Blue for scatter points

# Define event data
events = [
    (hitting_df['Human_Foot_Contact'], hitting_df['UPLIFT_Foot_Contact'], 
     'Hitting: Foot Contact', hit_fc_ccc, (0, 0)),
    (pitching_df['Human_Foot_Contact'], pitching_df['UPLIFT_Foot_Contact'], 
     'Pitching: Foot Contact', pitch_fc_ccc, (0, 1)),
    (hitting_df['Human_Ball_Contact'], hitting_df['UPLIFT_Ball_Contact'], 
     'Hitting: Ball Contact', hit_bc_ccc, (1, 0)),
    (pitching_df['Human_Release'], pitching_df['UPLIFT_Release'], 
     'Pitching: Release', pitch_rel_ccc, (1, 1)),
    (hitting_df['Human_Swing_Through'], hitting_df['UPLIFT_Swing_Through'], 
     'Hitting: Swing Through', hit_st_ccc, (2, 0)),
]

for human, auto, title, ccc_result, (row, col) in events:
    ax = axes[row, col]
    
    # Scatter plot
    ax.scatter(human, auto, alpha=0.6, s=50, c=SCATTER_COLOR, edgecolors='white', linewidth=0.5)
    
    # Get axis limits
    all_vals = np.concatenate([human.values, auto.values])
    min_val = all_vals.min() - 10
    max_val = all_vals.max() + 10
    
    # Line of perfect concordance (45° line)
    ax.plot([min_val, max_val], [min_val, max_val], 'k--', linewidth=2, label='Perfect Concordance')
    
    # Add regression line (best fit)
    z = np.polyfit(human, auto, 1)
    p = np.poly1d(z)
    x_line = np.linspace(min_val, max_val, 100)
    ax.plot(x_line, p(x_line), color=UPLIFT_PINK, linewidth=2, linestyle='-', label='Best Fit')
    
    # Set equal aspect ratio and limits
    ax.set_xlim(min_val, max_val)
    ax.set_ylim(min_val, max_val)
    ax.set_aspect('equal')
    
    # Labels and title
    ax.set_xlabel('Human Annotation (frames)', fontsize=14)
    ax.set_ylabel('UPLIFT Auto (frames)', fontsize=14)
    ax.set_title(title, fontweight='bold', fontsize=16)
    
    # Add CCC annotation box
    textstr = f'CCC = {ccc_result["CCC"]:.4f}\nr = {ccc_result["Pearson_r"]:.4f}\nCb = {ccc_result["Cb"]:.4f}\nn = {ccc_result["n"]}'
    props = dict(boxstyle='round', facecolor='white', alpha=0.9, edgecolor='gray')
    ax.text(0.05, 0.95, textstr, transform=ax.transAxes, fontsize=11,
            verticalalignment='top', bbox=props)
    
    ax.legend(loc='lower right', fontsize=10)
    ax.tick_params(axis='both', labelsize=12)
    ax.grid(True, alpha=0.3)

# Hide the empty subplot (bottom right)
axes[2, 1].axis('off')

plt.tight_layout()
plt.savefig('../figures/figure3_ccc_concordance.png', dpi=300, bbox_inches='tight')
print("✓ Saved: figure3_ccc_concordance.png")
plt.show()

### Figure 4: Bland-Altman Plots

In [ ]:
# ============================================================================# FIGURE 4: BLAND-ALTMAN PLOTS (Inter-Rater: Shun vs Ricky)# ============================================================================fig, axes = plt.subplots(2, 3, figsize=(15, 10))# Hitting eventshitting_events = [('Foot_Contact', 'Foot Contact'), ('Ball_Contact', 'Ball Contact'), ('Swing_Through', 'Swing Through')]for i, (col_name, display_name) in enumerate(hitting_events):    ax = axes[0, i]    shun = hitting_inter[f'Shun_{col_name}']    ricky = hitting_inter[f'Ricky_{col_name}']        mean_vals = (shun + ricky) / 2    diff_vals = shun - ricky        ax.scatter(mean_vals, diff_vals, alpha=0.7, s=60, c=UPLIFT_PINK, edgecolors='white')        # Mean and limits of agreement    mean_diff = diff_vals.mean()    sd_diff = diff_vals.std()    ax.axhline(mean_diff, color='blue', linestyle='-', linewidth=2, label=f'Mean: {mean_diff:.2f}')    ax.axhline(mean_diff + 1.96*sd_diff, color='red', linestyle='--', linewidth=1.5, label=f'+1.96 SD: {mean_diff + 1.96*sd_diff:.2f}')    ax.axhline(mean_diff - 1.96*sd_diff, color='red', linestyle='--', linewidth=1.5, label=f'-1.96 SD: {mean_diff - 1.96*sd_diff:.2f}')    ax.axhline(0, color='gray', linestyle=':', linewidth=1)        ax.set_xlabel('Mean (Shun + Ricky) / 2', fontsize=10)    ax.set_ylabel('Difference (Shun - Ricky)', fontsize=10)    ax.set_title(f'Hitting: {display_name}\n(n={len(diff_vals)})', fontsize=11, fontweight='bold')    ax.legend(fontsize=8, loc='upper right')# Pitching eventspitching_events = [('Foot_Contact', 'Foot Contact'), ('Release', 'Release')]for i, (col_name, display_name) in enumerate(pitching_events):    ax = axes[1, i]    shun = pitching_inter[f'Shun_{col_name}']    ricky = pitching_inter[f'Ricky_{col_name}']        mean_vals = (shun + ricky) / 2    diff_vals = shun - ricky        ax.scatter(mean_vals, diff_vals, alpha=0.7, s=60, c=UPLIFT_PINK, edgecolors='white')        mean_diff = diff_vals.mean()    sd_diff = diff_vals.std()    ax.axhline(mean_diff, color='blue', linestyle='-', linewidth=2, label=f'Mean: {mean_diff:.2f}')    ax.axhline(mean_diff + 1.96*sd_diff, color='red', linestyle='--', linewidth=1.5, label=f'+1.96 SD: {mean_diff + 1.96*sd_diff:.2f}')    ax.axhline(mean_diff - 1.96*sd_diff, color='red', linestyle='--', linewidth=1.5, label=f'-1.96 SD: {mean_diff - 1.96*sd_diff:.2f}')    ax.axhline(0, color='gray', linestyle=':', linewidth=1)        ax.set_xlabel('Mean (Shun + Ricky) / 2', fontsize=10)    ax.set_ylabel('Difference (Shun - Ricky)', fontsize=10)    ax.set_title(f'Pitching: {display_name}\n(n={len(diff_vals)})', fontsize=11, fontweight='bold')    ax.legend(fontsize=8, loc='upper right')# Hide empty subplotaxes[1, 2].axis('off')plt.suptitle('Figure 4: Bland-Altman Plots (Inter-Rater Agreement)', fontsize=14, fontweight='bold', y=1.02)plt.tight_layout()plt.savefig('../figures/figure4_bland_altman.png', dpi=150, bbox_inches='tight')plt.show()

## SummaryAll analyses complete. Key findings:- **CCC (Auto vs Human)**: All events show CCC > 0.99 (Almost Perfect agreement)- **ICC (Inter-Rater)**: All events show ICC > 0.99 (Excellent reliability)- **Error Distribution**: Mean errors within ±5 frames (±21 ms at 240 fps)- **Tolerance Analysis**: >90% of detections within ±12 frames of ground truth